In [14]:
from langgraph.graph import StateGraph, START, END 
from pydantic import BaseModel, Field 
from typing import TypedDict,Annotated
from dotenv import load_dotenv
import operator
from langchain_openai import ChatOpenAI

In [3]:
load_dotenv()

True

In [4]:
model = ChatOpenAI(model='gpt-4o-mini')

In [5]:
class EvaluationSchema(BaseModel):
    feedback : str = Field(description='Detailed feedback for the essay')
    score : int = Field(description='Score out of 10',ge=0,le=10)
    

In [6]:
structured_model = model.with_structured_output(EvaluationSchema)

In [7]:
essay = """

# Role of India in Artificial Intelligence (AI)

Artificial Intelligence (AI) has emerged as a transformative technology with the potential to reshape economies, governance, healthcare, education, and national security. As the world's most populous nation and one of the fastest-growing digital economies, India is uniquely positioned to play a significant role in the global AI landscape.

## India's Strengths in AI

India possesses several advantages that support its emergence as an AI powerhouse. First, it has a vast pool of skilled professionals in information technology, data science, and software engineering. The country's thriving IT industry and startup ecosystem provide a strong foundation for AI innovation.

Second, India's large population generates enormous volumes of digital data through initiatives such as Aadhaar, Unified Payments Interface (UPI), DigiLocker, and CoWIN. This digital public infrastructure creates opportunities for developing AI solutions tailored to large-scale governance and service delivery.

Third, India has become a global hub for startups, with numerous AI-driven enterprises working in healthcare, agriculture, fintech, education, and logistics.

## Government Initiatives

Recognizing the strategic importance of AI, the Government of India has launched several initiatives:

* National Strategy for Artificial Intelligence by NITI Aayog, focusing on "AI for All."
* IndiaAI Mission to strengthen computing infrastructure, datasets, innovation, and skill development.
* Digital India programme, which promotes digital connectivity and e-governance.
* Centres of Excellence in AI across sectors such as healthcare, agriculture, and education.
* Promotion of semiconductor manufacturing and indigenous technology development.

These initiatives aim to democratize AI benefits and ensure inclusive growth.

## India's Contribution to Development

AI can significantly contribute to India's developmental goals:

### Agriculture

AI-based solutions can assist farmers through crop monitoring, weather prediction, pest detection, and precision farming.

### Healthcare

AI can improve disease diagnosis, medical imaging, telemedicine, and healthcare accessibility in rural areas.

### Education

Personalized learning systems and AI-enabled educational tools can enhance learning outcomes and bridge educational inequalities.

### Governance

AI can improve public service delivery, policy formulation, grievance redressal, and administrative efficiency.

### Industry and Economy

AI can boost productivity, support manufacturing under Make in India, and strengthen India's competitiveness in the global economy.

## Challenges Before India

Despite its potential, India faces several challenges:

* Shortage of high-end AI research and advanced computing infrastructure.
* Concerns regarding data privacy and cybersecurity.
* Digital divide between urban and rural regions.
* Ethical issues such as algorithmic bias and transparency.
* Dependence on imported semiconductor technologies and advanced hardware.

Addressing these challenges is essential for sustainable AI development.

## Way Forward

India must focus on strengthening research and development, expanding digital infrastructure, promoting AI literacy, ensuring ethical governance, and encouraging public-private partnerships. Investments in indigenous semiconductor manufacturing, high-performance computing, and responsible AI frameworks will further enhance India's capabilities.

## Conclusion

Artificial Intelligence presents a historic opportunity for India to accelerate economic growth, improve governance, and achieve inclusive development. By leveraging its demographic dividend, digital public infrastructure, and innovation ecosystem, India can emerge as a global leader in responsible and human-centric AI while ensuring that the benefits of technology reach every section of society.


"""

In [8]:
prompt = f" Evaluate the language quality of the following essay and provide feedback and assign a score out of 10 \n {essay}"

In [12]:
response = structured_model.invoke(prompt)

In [13]:
print(response)

feedback='The essay presents a well-structured exploration of India\'s role in Artificial Intelligence, detailing its strengths, government initiatives, contributions to development, challenges, and the way forward. The use of clear headings improves readability and organization, allowing the reader to easily follow the argument. The language is formal yet accessible, making it suitable for an academic or professional audience.\n\nHowever, the essay could benefit from a few improvements:\n1. **Grammar and Syntax**: While the grammar is generally sound, there are a few awkward phrasings, such as the use of "AI-driven enterprises" which could be simplified to" AI companies" for clarity. \n2. **Detailing Examples**: Some sections, particularly in the challenges and government initiatives, would benefit from more specific examples or statistics to illustrate points more vividly. For instance, mentioning specific successful AI startups or projects would provide additional credibility. \n3. 

In [15]:
class UPSCState(TypedDict):
    essay : str
    language_feedback : str
    analysis_feedback : str
    clarity_feedback : str 
    overall_feedback : str 
    individual_scores : Annotated[list[int], operator.add]
    avg_score : float 


In [16]:
graph = StateGraph(UPSCState)

In [ ]:
def evaluate_language(state: UPSCState):
    prompt = f"Evaluate the language quality of the following essay and provide feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_model.invoke(prompt)
    return {'language_feedback':output.feedback,'individual_scores':[output.score]}

In [ ]:
def evaluate_analysis(state: UPSCState):
    prompt = f"Evaluate the depth of analysis of the following essay and provide feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_model.invoke(prompt)
    return {'analysis_feedback':output.feedback,'individual_scores':[output.score]}

In [ ]:
def evaluate_thought(state: UPSCState):
    prompt = f"Evaluate the clarity of thought of the following essay and provide feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_model.invoke(prompt)
    return {'clarity_feedback':output.feedback,'individual_scores':[output.score]}

In [ ]:
def final_evaluation(state:UPSCState):
    # Summary feedback 
    prompt=f"Based on the following feedbacks create a summarized feedback \n language feedback - {state['language_feedback']} \n Analysis feedback - {state['analysis_feedback']} \n clarity of thought feedback - {state['clarity_feedback']}"
    overall_feedback = model.invoke(prompt).content
    # avg calculate score 
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])
    return {'overall_feedback':overall_feedback, 'avg_score':avg_score}
    

In [17]:
graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)

NameError: name 'evaluate_language' is not defined